# Gene flow 


## Objectives

1. What is the D statistic?
2. How to calculate the D statistic & interpret the results


We will read in some simulated data (that includes 4 populations), extract the genotype matrix, filter to biallelic sites & then calculate the D statistic

In [ ]:
library(vcfR)

# read in simulated vcf
vcf <- read.vcfR("/course/kenya2026/harvi/geneflow/inputdata/hum_nea_siml.vcf.gz")
ingt <- extract.gt(vcf, element = "GT")

ingt[which(ingt=="0|0")]<-0
ingt[which(ingt=="0|1")]<-1
ingt[which(ingt=="1|0")]<-1
ingt[which(ingt=="1|1")]<-2

# how many loci
nrow(ingt)

#  unique(ingt[,1])
# [1] "0|0" "1|1" "1|0" "0|1" "2|0" "2|2" "1|2" "0|2" "1|3" "2|1"

# filter to biallelic sites only
biallelic <- ingt[!apply(ingt, 1, function(row) any(row %in% c("2|0", "2|2", "1|2", "0|2", "1|3", "2|1"))), ]

nrow(biallelic)

biallelic[which(biallelic=="0|0")]<-0
biallelic[which(biallelic=="0|1")]<-1
biallelic[which(biallelic=="1|0")]<-1
biallelic[which(biallelic=="1|1")]<-2

biallelicdf<-as.data.frame(biallelic)
# convert to numeric
biallelicdf[] <- lapply(biallelicdf, function(x) {if (is.character(x)) as.numeric(x) else x})

In [ ]:
# sample info, individuals of each population
sampleinfo <- read.table("/course/kenya2026/harvi/geneflow/inputdata/hum_nea_siml.tsv", header = T, sep = "\t")

outg <- sampleinfo[which(sampleinfo$pop == "CHIMP"),1]
p1 <- sampleinfo[which(sampleinfo$pop == "AFR"),1]
p2 <- sampleinfo[which(sampleinfo$pop == "EUR"),1]
p3 <- sampleinfo[which(sampleinfo$pop == "NEA"),1]


In [ ]:
# function to calc d statistic
calc_abba_baba <- function(ingt1) {
  
  f1 <- rowMeans(ingt1[, p1, drop = FALSE], na.rm = TRUE) / 2
  f2 <- rowMeans(ingt1[, p2, drop = FALSE], na.rm = TRUE) / 2
  f3 <- rowMeans(ingt1[, p3, drop = FALSE], na.rm = TRUE) / 2
  fo <- rowMeans(ingt1[, outg, drop = FALSE], na.rm = TRUE) / 2
  
  keep <- is.finite(f1) & is.finite(f2) & is.finite(f3) & is.finite(fo)
  
  abba <- sum((1 - f1[keep]) * f2[keep] * f3[keep])
  baba <- sum(f1[keep] * (1 - f2[keep]) * f3[keep])
  
  D <- (abba - baba) / (abba + baba)
  
return(cbind(abba = abba,baba = baba,D = D,n_sites = sum(keep)))}

calc_abba_baba(ingt1=biallelicdf)


### Questions (1)
1. Please interpret the table you generated in the previous code block.
2. Does it fit with what we learnt in the lecture?
3. If this was your analysis what would be your next step?

### Questions (2)

4. What does the D statistic detect?

A. Evidence of introgression or gene flow between populations; 
B. Genetic drift within a single population; 
C. The mutation rate of a population;  
D. Effective population size of a population

5. Why do we need an outgroup for the D statistic?

A. It identifies which population has the largest effective population size;
B. It provides the ancestral/reference allele state;
C. It identifies the population with the most introgression;
D. It determines the mutation rate

6. If the number of ABBA sites = 200 and number of BABA sites = 200, what is the value of D statistic?

A. -1;
B. -0.5;
C. 0;
D. 1

7. If the number of ABBA sites = 600 and number of BABA sites = 400, what is the value of D statistic?

A. -0.20;
B. 0;
C. 0.40;
D. 0.20;
Hint: D = (ABBA - BABA) / (ABBA + BABA)

8. If P1 and P3 had gene flow which site pattern would be in excess?
A. ABBA;
B. BABA;
C. AABB;
D. Neither